# Stylia demo

A walkthrough of all major features: named colors, categorical palettes, continuous colormaps, and figure utilities.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import stylia
from stylia import (
    NamedColors,
    CategoricalPalette,
    NamedColorMaps,
    ContinuousColorMap,
    create_figure,
    save_figure,
    FONTSIZE, FONTSIZE_BIG, MARKERSIZE, LINEWIDTH,
    ONE_COLUMN_WIDTH, TWO_COLUMNS_WIDTH,
)

rng = np.random.default_rng(42)

---
## 1. Named colors

`NamedColors` provides individual, semantically named colors — Ersilia brand colors first,
then NPG fill-ins for hues the brand palette does not cover.

In [ ]:
nc = NamedColors()

# Access by name
print("plum  :", nc.plum)
print("mint  :", nc.mint)
print("blue  :", nc.blue)

# Access by index
print("nc[0] :", nc[0])    # plum
print("nc[0:3]:", nc[0:3]) # plum, purple, mint

# Modifiers
print("plum 40% alpha:", nc.get('plum', alpha=0.4))
print("blue lightened:", nc.get('blue', lighten=0.3))

# All hex values
nc.hex

In [ ]:
# Visualise all named colors
from stylia.colors.colors import _COLOR_ORDER, _NAMED

fig, ax = plt.subplots(figsize=(ONE_COLUMN_WIDTH * 1.2, 3.6))
ax.set_facecolor('white'); ax.axis('off')

per_col = (len(_COLOR_ORDER) + 1) // 2
for ci, names in enumerate([_COLOR_ORDER[:per_col], _COLOR_ORDER[per_col:]]):
    for ri, name in enumerate(names):
        xc = 0.03 + ci * 0.49
        yc = 0.93 - ri * (0.82 / (per_col - 1))
        ax.add_patch(plt.Circle((xc + 0.045, yc), 0.03, color=nc.get(name),
                                transform=ax.transAxes, clip_on=False))
        ax.text(xc + 0.10, yc + 0.005, name, transform=ax.transAxes,
                fontsize=8, va='center', color='#2C3E50')
        ax.text(xc + 0.10, yc - 0.055, _NAMED[name], transform=ax.transAxes,
                fontsize=6.5, va='center', color='#8D99AE', fontfamily='monospace')

ax.set_title('Named Colors', fontsize=10, fontweight='bold', color='#2C3E50', loc='left', pad=10)
plt.tight_layout()
plt.show()

---
## 2. Categorical palettes

`CategoricalPalette` cycles through a set of distinct colors for grouped / categorical data.
Use `.sample(n)` to get n colors, `.next()` to draw one at a time.

In [ ]:
# Available presets
print(CategoricalPalette.available())

# Usage
pal = CategoricalPalette()          # npg by default
pal = CategoricalPalette('ersilia') # Ersilia brand
pal = CategoricalPalette('okabe')   # colorblind-safe

colors = pal.sample(5)              # list of 5 RGB tuples
color  = pal.next()                 # draw one
pal.reset()                         # restart counter
print('5 colors:', colors[:2], '...')

In [ ]:
# Visualise all presets as color dot rows
from stylia.colors.colors import _CATEGORICAL_PRESETS, _hex_to_rgb

preset_names = CategoricalPalette.available()
fig, ax = plt.subplots(figsize=(TWO_COLUMNS_WIDTH, 3.0))
ax.set_facecolor('white'); ax.axis('off')

row_h = 1.0 / (len(preset_names) + 1)
for ri, pname in enumerate(preset_names):
    y = 1.0 - (ri + 0.65) * row_h
    ax.text(0.01, y, f'{pname}', transform=ax.transAxes,
            fontsize=8, va='center', color='#2C3E50', fontfamily='monospace', fontweight='bold')
    colors = [_hex_to_rgb(c) for c in _CATEGORICAL_PRESETS[pname]]
    for ci, color in enumerate(colors):
        ax.add_patch(plt.Circle((0.23 + ci * 0.055, y), row_h * 0.28,
                                color=color, transform=ax.transAxes, clip_on=False))

ax.set_title('Categorical Palettes', fontsize=10, fontweight='bold', color='#2C3E50', loc='left', pad=10)
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart example
pal = CategoricalPalette('npg')
groups = ['Control', 'Treatment A', 'Treatment B', 'Treatment C', 'Treatment D']
values = [0.42, 0.71, 0.58, 0.86, 0.63]
colors = pal.sample(len(groups))

fig, axes = create_figure()
ax = axes[0]
ax.bar(groups, values, color=colors, width=0.6, edgecolor='none')
ax.set_ylabel('Value')
ax.set_title('NPG categorical palette')
ax.set_ylim(0, 1.05)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
## 3. Continuous colormaps

`NamedColorMaps` provides named access to perceptually-uniform colormaps.

All colormaps are chosen so that **every data value is visible on a white background**.
Diverging maps use a **dark center** (L ≈ 0.07–0.25) instead of the conventional white center,
so mid-range values are never invisible.

In [ ]:
ncm = NamedColorMaps()
print('Available:', ncm.available)

In [ ]:
# Show gradient bars for all colormaps
gradient = np.linspace(0, 1, 256).reshape(1, -1)

sections = {
    'Sequential': ['viridis', 'heat', 'ocean', 'earth'],
    'Diverging — dark center': ['spectral', 'coolwarm'],
    'Cyclic': ['cyclic'],
}

n_cmaps = sum(len(v) for v in sections.values())
fig_h = 0.38 * n_cmaps + 0.6 * len(sections) + 0.5
fig = plt.figure(figsize=(6.5, fig_h))
fig.patch.set_facecolor('white')

label_w_frac = 0.20
bar_x = label_w_frac
bar_w = 1.0 - label_w_frac - 0.02

row_h_frac  = 0.38 / fig_h
sec_h_frac  = 0.55 / fig_h
gap_frac    = 0.12 / fig_h

y_top = 1.0 - 0.08 / fig_h
for sec_name, names in sections.items():
    # section label axes
    tax = fig.add_axes([0.0, y_top - sec_h_frac, 1.0, sec_h_frac])
    tax.axis('off')
    tax.text(0.01, 0.5, sec_name, transform=tax.transAxes,
             fontsize=8.5, fontweight='bold', color='#2C3E50', va='center')
    y_top -= sec_h_frac

    for cname in names:
        cmap = ncm.get(cname)
        # gradient bar
        bax = fig.add_axes([bar_x, y_top - row_h_frac, bar_w, row_h_frac])
        bax.imshow(gradient, aspect='auto', cmap=cmap)
        bax.set_axis_off()
        # label
        lax = fig.add_axes([0.0, y_top - row_h_frac, label_w_frac, row_h_frac])
        lax.axis('off')
        lax.text(0.95, 0.5, cname, transform=lax.transAxes,
                 fontsize=8, color='#2C3E50', va='center', ha='right')
        y_top -= (row_h_frac + gap_frac)

    y_top -= 0.04 / fig_h

plt.show()

### Fitting a colormap to data

`ContinuousColorMap` applies a quantile transformation so colors spread evenly across the data distribution — important when data has skewed or non-uniform density.

In [ ]:
n = 400
x = rng.standard_normal(n)
y = rng.standard_normal(n)
z = x - y  # diverging around 0

ccm = ContinuousColorMap('spectral')  # dark-center diverging
ccm.fit(z)
c = ccm.transform(z)

fig, axes = create_figure(ncols=2)
ax0 = axes[0]
ax0.scatter(x, y, c=c, s=MARKERSIZE, linewidths=0, alpha=0.85)
ax0.set_title('spectral – diverging (dark center)')
ax0.set_xlabel('x'); ax0.set_ylabel('y')

z2 = np.sqrt(x**2 + y**2)
ccm2 = ContinuousColorMap('viridis')
ccm2.fit(z2)
ax1 = axes[1]
ax1.scatter(x, y, c=ccm2.transform(z2), s=MARKERSIZE, linewidths=0, alpha=0.85)
ax1.set_title('viridis – sequential')
ax1.set_xlabel('x'); ax1.set_ylabel('y')

plt.tight_layout()
plt.show()

In [ ]:
# All available options
data = rng.standard_normal(200)

ccm = ContinuousColorMap('spectral')                       # uniform quantile (default)
ccm = ContinuousColorMap('heat',    transformation='normal') # normal quantile
ccm = ContinuousColorMap('viridis', transformation=None)    # raw percentile clip
ccm = ContinuousColorMap('coolwarm', ascending=False)       # reversed direction

ccm.fit(data)
colors       = ccm.transform(data)           # list of RGBA tuples
colors_alpha = ccm.get(data, alpha=0.5)      # with alpha
swatches     = ccm.sample(8)                 # 8 evenly-spaced swatches
print(f'Transformed {len(colors)} points, first color: {colors[0]}')

---
## 4. Figures

`create_figure` returns `(fig, axes)` — axes are styled automatically when accessed.

In [ ]:
# Single panel
fig, axes = create_figure()
ax = axes[0]
ax.plot(np.linspace(0, 2*np.pi, 100), np.sin(np.linspace(0, 2*np.pi, 100)),
        color=nc.plum, linewidth=1.5)
ax.set_xlabel('x'); ax.set_ylabel('sin(x)'); ax.set_title('Single panel')
plt.tight_layout(); plt.show()

In [ ]:
# Multi-panel with named colors and categorical palette
pal = CategoricalPalette('npg')
t   = np.linspace(0, 2*np.pi, 200)

fig, axes = create_figure(ncols=3)

# Scatter
ax = axes[0]
for i, label in enumerate(['A', 'B', 'C', 'D']):
    xi = rng.standard_normal(50) + i * 0.4
    yi = rng.standard_normal(50)
    ax.scatter(xi, yi, color=pal.next(), s=10, label=label, linewidths=0)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_title('Scatter')
ax.legend(fontsize=FONTSIZE, markerscale=1.5)

# Line
ax = axes[1]
pal.reset()
for freq in [1, 2, 3, 4]:
    ax.plot(t, np.sin(freq * t), color=pal.next(), linewidth=1)
ax.set_xlabel('t'); ax.set_ylabel('amplitude'); ax.set_title('Line')

# Bar
ax = axes[2]
pal.reset()
cats = ['A', 'B', 'C', 'D', 'E']
vals = rng.uniform(0.3, 0.9, len(cats))
ax.bar(cats, vals, color=pal.sample(len(cats)), edgecolor='none')
ax.set_ylabel('Value'); ax.set_title('Bar')

plt.tight_layout()
plt.show()

---
## 5. Sizes and constants

Use the exported constants for consistent sizing across all figures.

In [ ]:
from stylia import (
    FONTSIZE_SMALL, FONTSIZE, FONTSIZE_BIG,
    MARKERSIZE_SMALL, MARKERSIZE, MARKERSIZE_BIG,
    LINEWIDTH, LINEWIDTH_THICK,
    ONE_COLUMN_WIDTH, TWO_COLUMNS_WIDTH,
)

print(f'Font sizes  : {FONTSIZE_SMALL}, {FONTSIZE}, {FONTSIZE_BIG} pt')
print(f'Marker sizes: {MARKERSIZE_SMALL}, {MARKERSIZE}, {MARKERSIZE_BIG}')
print(f'Line widths : {LINEWIDTH}, {LINEWIDTH_THICK}')
print(f'Figure widths: {ONE_COLUMN_WIDTH} in (1-col), {TWO_COLUMNS_WIDTH} in (2-col)')